# Basic Usage: ORMAS-CI on Water

This notebook demonstrates the core workflow of the `ormas_ci` package:
partitioning an active orbital space into subspaces with occupation constraints,
running ORMAS-CI through PySCF's CASCI interface, and comparing the results
against a full CASCI reference calculation.

We will use the water molecule (H2O) as a test system, partitioning the active
orbitals into sigma and pi subspaces. This is a small enough system to run
quickly while still illustrating the key concepts:

- How to define `Subspace` objects with occupation bounds
- How to assemble an `ORMASConfig` and plug it into PySCF
- How to compare energies and determinant counts against full CASCI
- How to extract natural orbital occupations from the 1-RDM

In [ ]:
import numpy as np
from pyscf import gto, scf, mcscf
from ormas_ci import ORMASFCISolver, ORMASConfig, Subspace
from ormas_ci.determinants import count_determinants, casci_determinant_count

## Setting up the molecule

We build a water molecule in the cc-pVTZ basis and run a restricted Hartree-Fock
calculation. The RHF orbitals provide the starting point for the active space
methods that follow. We set `verbose=0` to suppress PySCF's detailed output.

In [ ]:
mol = gto.M(
    atom='O 0 0 0; H 0 0.757 0.587; H 0 -0.757 0.587',
    basis='cc-pvtz',
    verbose=0,
)
mf = scf.RHF(mol)
mf.verbose = 0
mf.run()
print(f"RHF energy: {mf.e_tot:.10f} Hartree")

## Running full CASCI for reference

Before applying any occupation restrictions, we run a full CASCI calculation
to establish a reference energy. We choose an active space of 4 orbitals and
6 electrons (3 alpha, 3 beta). This represents the valence sigma and pi
orbitals of water.

Full CASCI includes all possible determinants within this active space.
Our ORMAS-CI result should be at or slightly above this energy.

In [ ]:
ncas, nelecas = 4, (3, 3)

mc_ref = mcscf.CASCI(mf, ncas, nelecas)
mc_ref.verbose = 0
e_ref = mc_ref.kernel()[0]
print(f"CASCI energy: {e_ref:.10f} Hartree")

## Defining ORMAS subspaces

Now we partition the 4 active orbitals into two subspaces:

- **sigma** (orbitals 0, 1): The O-H sigma bonding and antibonding orbitals.
  We allow between 2 and 4 electrons in this subspace.
- **pi** (orbitals 2, 3): The lone pair / pi-type orbitals.
  We also allow between 2 and 4 electrons here.

These constraints limit inter-subspace excitations: electrons can rearrange
freely within each subspace, but only a limited number of electrons can
transfer between sigma and pi. This reduces the determinant count while
retaining the most physically important configurations.

The `ORMASConfig` bundles the subspace definitions together with the total
active space size and electron count. We then create an `ORMASFCISolver`
and plug it into PySCF's CASCI as a drop-in replacement for the default
FCI solver.

In [ ]:
config = ORMASConfig(
    subspaces=[
        Subspace("sigma", [0, 1], min_electrons=2, max_electrons=4),
        Subspace("pi", [2, 3], min_electrons=2, max_electrons=4),
    ],
    n_active_orbitals=ncas,
    nelecas=nelecas,
)

mc = mcscf.CASCI(mf, ncas, nelecas)
mc.verbose = 0
mc.fcisolver = ORMASFCISolver(config)
e_ormas = mc.kernel()[0]
print(f"ORMAS-CI energy: {e_ormas:.10f} Hartree")

## Comparing results

Let us compare the ORMAS-CI energy against the full CASCI reference, and
examine how many determinants each method uses. The ORMAS energy should be
at or slightly above the CASCI energy (it is variational). The determinant
count tells us how much computational savings we achieve.

In [ ]:
n_ormas = count_determinants(config)
n_casci = casci_determinant_count(ncas, nelecas)
energy_diff_mh = (e_ormas - e_ref) * 1000

print("Energy comparison")
print("=" * 45)
print(f"  CASCI energy:    {e_ref:.10f} Hartree")
print(f"  ORMAS-CI energy: {e_ormas:.10f} Hartree")
print(f"  Difference:      {energy_diff_mh:+.4f} mHartree")
print()
print("Determinant counts")
print("=" * 45)
print(f"  CASCI:    {n_casci:>6d} determinants")
print(f"  ORMAS-CI: {n_ormas:>6d} determinants")
print(f"  Reduction: {n_ormas/n_casci:.1%} of full CASCI space")
print(f"  Removed:   {n_casci - n_ormas} determinants ({(1 - n_ormas/n_casci):.1%})")

## Natural orbital occupations

The one-particle reduced density matrix (1-RDM) from the ORMAS-CI calculation
contains information about orbital occupations. Its eigenvalues are the natural
orbital occupation numbers, which range from 0 (empty) to 2 (doubly occupied).

Values near 0 or 2 indicate orbitals that are mostly inactive in the
correlation treatment. Values significantly different from 0 or 2 indicate
strong multireference character -- these are the orbitals that matter most.

In [ ]:
rdm1 = mc.fcisolver.make_rdm1(mc.ci, ncas, nelecas)
occupations = np.linalg.eigvalsh(rdm1)[::-1]

print("Natural orbital occupations")
print("=" * 35)
for i, occ in enumerate(occupations):
    print(f"  Orbital {i}: {occ:.6f}")
print(f"  Sum:       {occupations.sum():.6f} (should equal {sum(nelecas)})")

## Summary

In this notebook we demonstrated the basic ORMAS-CI workflow:

1. Built a molecule and ran RHF with PySCF.
2. Ran full CASCI as a reference.
3. Defined ORMAS subspaces with occupation constraints and ran ORMAS-CI
   as a drop-in FCI solver replacement.
4. Compared energies and determinant counts to quantify the accuracy/cost tradeoff.
5. Extracted natural orbital occupations from the 1-RDM.

The key takeaway is that ORMAS-CI can significantly reduce the number of
determinants while introducing only a small energy error, provided the
subspace partitioning reflects the physical structure of the problem.
For larger active spaces, the determinant reduction can be dramatic.